The Main Data Loading Pipeline Summarized
The complete chapter code is located in ch02.ipynb.

This notebook contains the main takeaway, the data loading pipeline without the intermediate steps.

Packages that are being used in this notebook:

In [ ]:
# 中文注释：以下这一行只是一个注释性质的“章节标题”，用于标记本 notebook 主体代码的起点，本身不包含可执行逻辑
#The Main Data Loading Pipeline Summarized

In [ ]:
# NBVAL_SKIP
# 中文注释：打印当前环境中 torch 与 tiktoken 的版本号，便于复现实验环境（NBVAL_SKIP 表示自动化测试工具 nbval 会跳过本单元的输出校验）
from importlib.metadata import version

print("torch version:", version("torch"))  # 打印 PyTorch 版本
print("tiktoken version:", version("tiktoken"))  # 打印 tiktoken（GPT-2 所用的 BPE 分词器库）版本

In [ ]:
import tiktoken  # 导入 tiktoken：OpenAI 开源的 BPE 分词器库，用于将文本编码为 token id 序列
import torch  # 导入 PyTorch，用于张量运算，并构建自定义数据集 / 数据加载器
from torch.utils.data import DataLoader,Dataset  # DataLoader 负责按批次加载样本；Dataset 是自定义数据集需要继承的基类
class GPTDatasetV1(Dataset):
    # 自定义数据集：用“滑动窗口”把整段长文本切分成若干 (输入, 目标) 样本对，
    # 用于训练 GPT 这类自回归语言模型“根据上文预测下一个 token”的任务
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids=[]  # 保存所有输入片段：每个片段是长度为 max_length 的 token id 序列（张量）
        self.target_ids=[]  # 保存与输入片段一一对应的目标片段（即输入整体向右错位一位后的 token id 序列）
        token_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})  # 把整篇文本一次性编码成一长串 token id；允许文本中出现特殊标记 <|endoftext|>
        # 滑动窗口切分：
        # - 窗口长度固定为 max_length
        # - 每次向右滑动 stride 个 token（stride 越小，相邻窗口重叠越多，切分出的样本数越多）
        # - range 上限用 len(token_ids)-max_length，确保最后一个窗口（以及它错位一位后的 target）不会超出 token_ids 的边界
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i+max_length]  # 输入片段：从位置 i 开始，连续取 max_length 个 token
            target_chunk=token_ids[i+1:i+max_length+1]  # 目标片段：与输入片段相比整体右移 1 个 token（错位一位），
            # 也就是说 target 每个位置上的 token，正好是 input 对应位置 token 的“下一个词”，
            # 这正是自回归语言模型训练所需要的 (输入序列, 下一个词) 监督信号
            self.input_ids.append(torch.tensor(input_chunk))  # 转成张量后保存，形状为 [max_length]
            self.target_ids.append(torch.tensor(target_chunk))  # 转成张量后保存，形状同样为 [max_length]
    def __len__(self):
        return len(self.input_ids)  # 数据集大小 = 滑动窗口切分出来的样本（窗口）总数
    def __getitem__(self, ids):
        return self.input_ids[ids],self.target_ids[ids]  # 按索引取出一对 (输入张量, 目标张量)，二者形状均为 [max_length]

def create_dataloader_v1(txt,batch_size,max_length,stride,
                         shuffle=True,drop_last=True,num_workers=0
                         ):
    # 封装函数：完成 原始文本 -> 分词编码 -> 滑动窗口切分数据集 -> 封装为 PyTorch DataLoader 的完整流程
    tokenizer=tiktoken.get_encoding("gpt2")  # 使用 GPT-2 对应的 BPE 分词器（词表大小为 50257）
    dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)  # 构建滑动窗口数据集，得到若干 (输入, 目标) 样本对
    dataloader=DataLoader(
        dataset,
        shuffle=shuffle,  # 是否打乱样本顺序（训练阶段通常设为 True）
        batch_size=batch_size,  # 每个 batch 包含多少个 (输入, 目标) 样本对
        drop_last=drop_last,  # 是否丢弃最后一个样本数不足 batch_size 的不完整批次
        num_workers=num_workers  # 数据加载所用的子进程数，0 表示在主进程中同步加载
    )
    return dataloader

# 读取原始训练语料：这里使用短篇小说《The Verdict》全文（文件名没有扩展名，但内容为纯文本），作为演示用的小型语料库
with open("the-verdict",'r',encoding='utf-8') as f :
    raw_text=f.read()
vocab_size=50257  # GPT-2 分词器的词表大小，需与嵌入表的行数一致
output_dim=256  # 每个 token 的嵌入向量维度（此处为演示用的较小维度，真实 GPT-2 模型维度更大，如 768/1024 等）
context_length=1024  # 模型支持的最大上下文长度，也是位置嵌入表需要覆盖的最大位置数
token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)  # 词元嵌入表：将每个 token id 映射为一个 output_dim 维的向量
pos_embedding_layer=torch.nn.Embedding(context_length,output_dim)  # 位置嵌入表：将每个“位置索引”（0,1,2,...）映射为一个 output_dim 维的向量，用于向模型注入顺序信息
batch_size = 8  # 每个 batch 中包含的样本（窗口）数量
max_length = 4  # 滑动窗口长度，即每个输入/目标序列所包含的 token 数量
dataloader=create_dataloader_v1(
    raw_text,
    batch_size=batch_size,
    max_length=max_length,
    stride=max_length
)


In [ ]:
for  batch in dataloader:  # 从 DataLoader 中取出第一个 batch 用于演示（配合下面的 break 只跑一次）
    x,y=batch  # x: 输入 token id，形状 [batch_size, max_length]；y: 目标 token id，形状同样为 [batch_size, max_length]（本单元未使用 y）
    token_embeddings=token_embedding_layer(x)  # 查词元嵌入表，得到形状 [batch_size, max_length, output_dim] 的张量
    pos_embeddings=pos_embedding_layer(torch.arange(max_length))  # 查位置嵌入表，得到形状 [max_length, output_dim] 的张量（对 batch 内所有样本通用，广播使用）
    # 【bug 修复】原代码此处为 token_embeddings-pos_embeddings（减法），这是一个错误：
    # Transformer/GPT 结构中，词元嵌入应与位置嵌入相加（而非相减）来注入顺序信息，
    # 这里改为加法 + ，以匹配《Build a Large Language Model From Scratch》原书的标准做法
    input_embeddings=token_embeddings+pos_embeddings  # 广播相加：[batch_size, max_length, output_dim] + [max_length, output_dim] -> [batch_size, max_length, output_dim]
    break  # 只演示第一个 batch，随后跳出循环

In [ ]:
print(input_embeddings.shape)  # 打印输入嵌入张量的形状，预期为 [batch_size, max_length, output_dim] = [8, 4, 256]